# Tutorial 2 — From present-day glacier to future projections

At the end of Tutorial 1 we created a Level 3 glacier directory containing:

- glacier geometry
- climate data
- calibrated mass balance model
- inferred ice thickness
- dynamic flowlines

If you are happy with the standard OGGM setup, you do not need to **re-run** all Levels 0–3 tasks yourself. You can forget about the first tutorial.

Instead, you can download an already prepared Level 3 glacier directory:

In [ ]:
from oggm import cfg, utils, workflow, tasks, DEFAULT_BASE_URL

import geopandas as gpd
import numpy as np
import os

# we always need to initialise and define a working directory
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='OGGM-full_prepro_elevation_bands', reset=True)

# Our example glacier
rgi_ids = ['RGI60-11.00897']  # Hintereisferner
rgi_region = '11'  # this must fit to example glacier(s), if starting from level 0

    
gdirs = workflow.init_glacier_directories(rgi_ids,
                                          from_prepro_level=3,
                                          prepro_base_url=DEFAULT_BASE_URL,
                                          prepro_border=80,  # could be 10, 80, 160 or 240
                                          reset=True,
                                          force=True,
                                         )

In [ ]:
os.listdir(gdirs[0].dir)

## Level 4 — Dynamic initialization

One challenge in glacier modelling is that we usually do not know the past state of a glacier. We may know the glacier outline at one point in time, and we may have observations of recent glacier change, such as geodetic mass balance, but the full glacier evolution before the inventory date is unknown.

In addition, the datasets used to build a glacier directory often represent different points in time. For example, glacier outlines, DEMs, climate datasets, geodetic mass-balance observations, and consensus ice-volume estimates may all correspond to different years or averaging periods. The glacier state created in Tutorial 1 therefore combines valuable information from multiple sources, but it does not necessarily represent a dynamically consistent glacier at a single point in time.

The glacier state created in Tutorial 1 is therefore a reasonable approximation of the glacier *at the inventory date*, but it is not yet ideal for future projections.

So before running future projections, OGGM tries to answer this question:

<table>
<tr>
    <td>1979</td>
    <td>2003</td>
</tr>  
<tr>
    <td>? ───────► model ───────► </td>
    <td> observed glacier </td>
</tr>  
</table>

**What happened to this glacier before today?**

This is the purpose of Level 4.

Level 4 uses historical climate data and recent observations to reconstruct a physically plausible glacier evolution before the inventory date. This produces a glacier that is more dynamically consistent and therefore better suited for future projections.

**Two possible approaches**

### Historical simulation (legacy approach)

Before OGGM v1.6, historical simulations typically started directly from the Level 3 glacier state:

In [ ]:
# Level 4 — Dynamic initialization
# We use elevation-band flowlines only in this tutorial.

# Set the ice dynamic solver for elevation-band flowlines
cfg.PARAMS['evolution_model'] = 'SemiImplicit'

# Get the start and end year of the selected baseline climate period
y0 = gdirs[0].get_climate_info()['baseline_yr_0']
ye = gdirs[0].get_climate_info()['baseline_yr_1'] + 1


print('Starting year', y0)
print('Finish year', ye)

# Optional: historical run without dynamic spinup
# This was the default method until OGGM v1.6 and is mainly useful for comparison.
workflow.execute_entity_task(
    tasks.run_from_climate_data,
    gdirs,
    min_ys=y0,
    ye=ye,
    fixed_geometry_spinup_yr=None,
    store_fl_diagnostics=True,
    output_filesuffix='_historical'
)

This method is mainly useful for comparison because it does not attempt to reconstruct the glacier state prior to the inventory date.

In [ ]:
os.listdir(gdirs[0].dir)

The output of that run is `model_diagnostics_historical.nc`

### Dynamic initialization (recommended)

The standard OGGM workflow uses a dynamic spinup combined with a dynamic `melt_f` calibration:

In [ ]:
# Dynamic initialization, including dynamic melt_f calibration
dynamic_spinup_start_year = 1979
minimise_for = 'area'

workflow.execute_entity_task(
    tasks.run_dynamic_melt_f_calibration,
    gdirs,
    err_dmdtda_scaling_factor=0.2,
    ys=dynamic_spinup_start_year,
    ye=ye,
    kwargs_run_function={'minimise_for': minimise_for,
                         'store_fl_diagnostics': True},
    ignore_errors=True,
    kwargs_fallback_function={'minimise_for': minimise_for,
                              'store_fl_diagnostics': True},
    output_filesuffix='_spinup_historical',
)

This workflow:

- Starts from the Level 3 glacier state.
- Reconstructs a plausible glacier state in the recent past.
- Simulates glacier evolution under historical climate.
- Refines the previously calibrated `melt_f` *while accounting for changing glacier geometry*. In Level 3, the mass-balance calibration was done assuming fixed glacier geometry.
- Produces an improved glacier state for future simulations.

In the standard OGGM setup, the dynamic spinup attempts to match glacier **area**, because area is directly observed from glacier outlines, whereas volume is estimated from models.

In [ ]:
os.listdir(gdirs[0].dir)

Now OGGM creates more files in the glacier directory. The table below helps explain which ones are useful for the future simulations.

| File                                     | What does it represent?                                                                                                                                                        |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `model_flowlines.pkl`                    | **Level 3 glacier state**: the best estimate of the glacier at the inventory date, after climate calibration, inversion, and creation of dynamic flowlines.                    |
| `model_diagnostics_historical.nc`        | **Historical run without dynamic spinup**: summary time series such as area, volume, length, mass balance, and runoff when the model starts directly from the Level 3 glacier. |
| `model_geometry_spinup_historical.nc`    | **Glacier geometry during dynamic spinup**: the evolving glacier shape through time, including surface elevation, thickness, and flowline geometry.                            |
| `model_diagnostics_spinup_historical.nc` | **Diagnostics from the dynamic spinup run**: summary time series such as area, volume, length, mass balance, and runoff for the dynamically initialized glacier.               |
| `model_flowlines_dyn_melt_f_calib.pkl`   | **Dynamically calibrated glacier state**: the updated glacier flowlines after dynamic `melt_f` calibration, used as the improved starting point for future simulations.        |


### Compare Area and Volume

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

gdir = gdirs[0]

f = gdir.get_filepath('model_diagnostics', filesuffix='_historical')
with xr.open_dataset(f) as ds:
    ds_no_spinup = ds.load()

f = gdir.get_filepath('model_diagnostics', filesuffix='_spinup_historical')
with xr.open_dataset(f) as ds:
    ds_spinup = ds.load()

ds_no_spinup.area_m2.plot(label='No dynamic spinup')
ds_spinup.area_m2.plot(label='With dynamic spinup')

plt.title('Glacier area evolution')
plt.ylabel('Area (m²)')
plt.legend()

In [ ]:
ds_no_spinup.volume_m3.plot(label='No dynamic spinup')
ds_spinup.volume_m3.plot(label='With dynamic spinup')

plt.title('Glacier volume evolution')
plt.ylabel('Volume (m³)')
plt.legend()

### Dynamic spinup helps avoid initial shock: 

Artificial changes in glacier length, area, or velocity that can occur when a simulation starts directly from an **equilibrium glacier state**. Or directly from level 3 geometry.

A useful way to see this is to compare ice velocity along the flowline. The plot shows velocities in 2005 for two simulations: one starting directly from the Level 3 glacier, and one using dynamic spinup. The spinup run should give a smoother, more physically consistent velocity field because the glacier has already adjusted during the historical simulation.

In [ ]:
f = gdir.get_filepath('fl_diagnostics', filesuffix='_historical')
with xr.open_dataset(f, group='fl_0') as ds:
    dg_no = ds.load()

f = gdir.get_filepath('fl_diagnostics', filesuffix='_spinup_historical')
with xr.open_dataset(f, group='fl_0') as ds:
    dg_spin = ds.load()

year = 2005

dg_no.ice_velocity_myr.sel(time=year).plot(label='No spinup')
dg_spin.ice_velocity_myr.sel(time=year).plot(label='With spinup')

plt.title(f'Ice velocity along the flowline in {year}')
plt.ylabel('Ice velocity (m yr⁻¹)')
plt.legend()

Related Tutorials:

- [dynamical_spinup](https://tutorials.oggm.org/stable/notebooks/tutorials/dynamical_spinup.html): a deeper dive into the dynamical spinup for past simulations

- [numeric_solvers](https://tutorials.oggm.org/stable/notebooks/tutorials/numeric_solvers.html): Understand the difference between the ice dynamic solvers in OGGM

## Level 5 — Projection-ready glacier directories

So far we have retained a lot of information in the glacier directory, probably more than we actually need. If we are only interested in volume change, SLR contribution or runoff, we actually do not need all the files in the glacier directory in order to simulate a projection with a climate scenario. 

In [ ]:
os.listdir(gdirs[0].dir)

Level 5 can be confusing at first, because it **does not yet run a future projection**. Instead, it **prepares the glacier directory** so that future projections can be run efficiently.

After Level 4, the glacier has already been dynamically initialized. At this point, many files used during preprocessing are no longer needed for projection experiments. Level 5 keeps only the information required to run the glacier forward in time and stores it in a lighter glacier directory.

In [ ]:
mini_base_dir = os.path.join(cfg.PATHS['working_dir'], 'mini_per_glacier')

mini_gdirs = workflow.execute_entity_task(
    tasks.copy_to_basedir,
    gdirs,
    base_dir=mini_base_dir,
    setup='run/spinup'
)

Lets check how our working directory changed, and reduce the number of files in our  glacier dir `RGI60-11.00897`

In [ ]:
cfg.PATHS['working_dir']

In [ ]:
print('On the path above we have two folders now')
os.listdir(cfg.PATHS['working_dir'])

In [ ]:
os.listdir(os.path.join(cfg.PATHS['working_dir'], 'mini_per_glacier/RGI60-11/RGI60-11.00/RGI60-11.00897'))

In [ ]:
os.listdir(os.path.join(cfg.PATHS['working_dir'], 'per_glacier/RGI60-11/RGI60-11.00/RGI60-11.00897'))

#### How to use level 4-5?

Similar to previous tutorials we dont need to re-compute levels 0-5, we can just download them from the OGGM server. 

> **Important:** Levels 0 - 5 need to be re-computed **ONLY IF YOU ARE NOT** using standard OGGM inputs (Default outlines, Default DEM, GSWP3_W5E5 climate or custom MB or ice thickness measurements for calibration of MB model and ice dynamics.

In [ ]:
load_from_prepro_base_url = False 
load_from_prepro_base_url

In [ ]:
# Instruction for beginning with existing OGGM's preprocessed directories
if load_from_prepro_base_url:
    prepro_base_url_L3 = DEFAULT_BASE_URL
    gdirs = workflow.init_glacier_directories(rgi_ids,
                                              from_prepro_level=5,
                                              prepro_base_url=DEFAULT_BASE_URL,
                                              prepro_border=80,
                                              reset=True,
                                              force=True)

### Start from an existing glacier directory

Now we will finally do a future simulation using the `mini_per_glacier` directory created in Level 5.

This step is mainly for teaching, shows that you do not always need to rebuild a glacier directory from the beginning. Once a glacier has already been preprocessed, you can reload it and continue with a new experiment, such as a future projection.

You can also use the original `per_glacier` directory if you prefer. Here, we use `mini_per_glacier` to show how a lightweight Level 5 glacier directory can be reused for future simulations.

**Remember what ever directory you decide to use, it must have the name `per_glacier`**

So in order to use the new compressed directory we have to rename it.

In [ ]:
wd = cfg.PATHS['working_dir']
wd

In [ ]:
# Rename original full directory
os.rename(
    os.path.join(wd, 'per_glacier'),
    os.path.join(wd, 'per_glacier_full')
)

# Rename mini directory to become the active per_glacier
os.rename(
    os.path.join(wd, 'mini_per_glacier'),
    os.path.join(wd, 'per_glacier')
)

In [ ]:
from oggm import cfg, utils, workflow, tasks, DEFAULT_BASE_URL

import geopandas as gpd
import numpy as np
import os

# we always need to initialise and define a working directory
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = wd

# Our example glacier
rgi_ids = ['RGI60-11.00897']  # Hintereisferner
rgi_region = '11'  # this must fit to example glacier(s), if starting from level 0

gdirs = workflow.init_glacier_directories(rgi_ids)

In [ ]:
os.listdir(gdirs[0].dir)

## Future climate scenario

We now have a **projection-ready glacier directory**.

The Level 5 directory contains the dynamically initialized glacier state and the historical `_spinup_historical` run. This means we can now stop thinking about preprocessing and start asking:

### How might this glacier evolve under future climate change?

To run a future projection, OGGM needs two things:

1. a projection-ready glacier directory, such as Level 5 from `DEFAULT_BASE_URL`
2. future climate data from a Global Climate Model (GCM)

OGGM then uses the GCM climate scenario to simulate glacier evolution into the future.

In [ ]:
DEFAULT_BASE_URL

### Visualise our historical run.

We reconstruct our glacier under the following climate:

In [ ]:
cfg.PARAMS['baseline_climate']

In [ ]:
ds = utils.compile_run_output(gdirs, input_filesuffix='_spinup_historical')
vol_ref2000 = ds.volume / ds.volume.sel(time=2000) * 100
vol_ref2000.plot(hue='rgi_id')
plt.ylabel('Volume (%, reference 2000)');

In [ ]:
gdirs[0].rgi_date

The glacier volume and area estimates before that date are highly uncertain and serve the purpose of spinup only!


### Download and process GCM data from ISIMIP3b (bias-corrected CMIP6)

Choose one GCM member and future scenarios

In [ ]:
from oggm.shop import gcm_climate

member = 'mri-esm2-0_r1i1p1f1'
scenarios = ['ssp126', 'ssp370', 'ssp585']

# Process future climate data and run projections
for ssp in scenarios:
    rid = f'_ISIMIP3b_{member}_{ssp}'

    workflow.execute_entity_task(
        gcm_climate.process_monthly_isimip_data,
        gdirs,
        ssp=ssp,
        member=member,
        output_filesuffix=rid,
    )

How many scenarios are we running? 3, so we have 3 new climate files:

In [ ]:
import glob
glob.glob(os.path.join(gdirs[0].dir, 'gcm_data_*'))

#### Process future climate data and run projections

Now we will run the future simulation (Finally!) and show how we can compile a simulation into a netcdf file that stores runoff, thickness evolution and volume. 

In [ ]:
from oggm.sandbox import distribute_2d

# Optional but useful if you want model geometry stored for later diagnostics
cfg.PARAMS['store_model_geometry'] = True

In [ ]:
for ssp in scenarios:
    rid = f'_ISIMIP3b_{member}_{ssp}'

    # Future run with monthly runoff output
    workflow.execute_entity_task(
        tasks.run_with_hydro,
        gdirs,
        run_task=tasks.run_from_climate_data,
        climate_filename='gcm_data',
        climate_input_filesuffix=rid,
        init_model_filesuffix='_spinup_historical',
        store_monthly_hydro=True,
        ref_area_from_y0=True,
        output_filesuffix=rid,
    )

In [ ]:
for ssp in scenarios:
    rid = f'_ISIMIP3b_{member}_{ssp}'
    ds = utils.compile_run_output(gdirs, input_filesuffix=rid)
    ds.volume.sum(dim='rgi_id').plot(label=ssp)

plt.title('Projected glacier volume')
plt.ylabel('Volume (m³)')
plt.xlabel('Year')
plt.legend()
plt.show()

In [ ]:
gdir = gdirs[0]

runoff_vars = [
    'melt_off_glacier',
    'melt_on_glacier',
    'liq_prcp_off_glacier',
    'liq_prcp_on_glacier']

f, ax = plt.subplots(figsize=(12, 5))

for ssp in scenarios:
    rid = f'_ISIMIP3b_{member}_{ssp}'

    with xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=rid)) as ds:
        ds = ds.isel(time=slice(0, -1)).load()

    # Select annual runoff variables and convert kg to megatonnes
    runoff = ds[runoff_vars].to_array().sum(dim='variable').clip(0) * 1e-9

    # Smooth with an 11-year rolling mean
    runoff.rolling(time=11, center=True).mean().plot(ax=ax, label=ssp)

ax.set_title(f'Projected annual runoff: {gdir.rgi_id}')
ax.set_ylabel('Annual runoff (Mt)')
ax.set_xlabel('Year')
ax.legend()
plt.show()

For Hintereisferner, runoff decreases throughout the 21st-century for all scenarios, indicating that peak water has already been reached sometime in the past or is very close to be reached. This is the case for many European glaciers. What about our unnamed glacier in the Himalayas? Lets change the RGIID and repeat this tutorial to see if those glaciers have reach peak water.

**'RGI60-14.23809'**